In [34]:
import pandas as pd
import numpy as np
import os
import glob
import re
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
import plotly.subplots as sp
 

pd.set_option("display.max_columns", 100)

DATA_HOT_SCORE = Path("data/hotscore")
OUTPUT_DIR = Path("output/volatility")
OUTPUT_DATA_DIR = Path("data/volatility")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DATA_DIR.mkdir(parents=True, exist_ok=True)

In [35]:
def load_all_hotscore_files(directory=DATA_HOT_SCORE):
    files = sorted(glob.glob(str(directory / "hotscore_*.csv")))
    
    dfs = []
    
    for f in files:
        df = pd.read_csv(f)
        
        # extract date from filename
        date_str = os.path.basename(f).split("_")[1].replace(".csv", "")
        df["date"] = pd.to_datetime(date_str, format="%Y%m%d")
        
        dfs.append(df)
    
    return pd.concat(dfs, ignore_index=True)

df = load_all_hotscore_files()

df = df.sort_values(["symbol", "date"])
df.head(5)

,symbol,HotScore,TrendScore,regularMarketPrice,regularMarketChangePercent,VolumeSpike,averageDailyVolume3Month,MomentumScore,VolumeScore,VolatilityScore,marketCap,date
717500,A,0.943797,0.843672,138.450,19.518301,1.793232,2151527.0,0.985112,0.923077,0.957816,3.912629e+10,2026-05-28
717549,A,0.938902,0.827103,135.380,16.868100,2.574832,2151527.0,0.976636,0.927570,0.948598,3.825870e+10,2026-05-28
0,AA,0.794401,0.520833,41.845,6.747450,0.940394,6727448.0,0.903646,0.802083,0.726562,1.083635e+10,2026-01-17
50,AA,0.794401,0.520833,41.845,6.747450,0.940394,6727448.0,0.903646,0.802083,0.726562,1.083635e+10,2026-01-17
100,AA,0.773989,0.557951,41.560,6.020410,1.039631,6727448.0,0.876011,0.749326,0.746631,1.076255e+10,2026-01-17


In [36]:
df.info()

<class 'pandas.DataFrame'>
Index: 718055 entries, 717500 to 708639
Data columns (total 12 columns):
 #   Column                      Non-Null Count   Dtype         
---  ------                      --------------   -----         
 0   symbol                      718055 non-null  str           
 1   HotScore                    715634 non-null  float64       
 2   TrendScore                  715634 non-null  float64       
 3   regularMarketPrice          715634 non-null  float64       
 4   regularMarketChangePercent  715634 non-null  float64       
 5   VolumeSpike                 715634 non-null  float64       
 6   averageDailyVolume3Month    715634 non-null  float64       
 7   MomentumScore               715634 non-null  float64       
 8   VolumeScore                 715634 non-null  float64       
 9   VolatilityScore             715634 non-null  float64       
 10  marketCap                   715631 non-null  float64       
 11  date                        718055 non-null  datet

In [37]:
price_stats = df.groupby("symbol").agg(
    min_price=("regularMarketPrice", "min"),
    max_price=("regularMarketPrice", "max"),
    mean_price=("regularMarketPrice", "mean")
)

price_stats["range_pct"] = (
    (price_stats["max_price"] - price_stats["min_price"]) 
    / price_stats["mean_price"]
)

In [38]:
df["direction"] = np.sign(df["regularMarketChangePercent"])

direction_changes = (
    df.groupby("symbol")["direction"]
    .apply(lambda x: (x.diff() != 0).sum())
    .to_frame(name="direction_changes")
)

In [39]:
metrics = df.groupby("symbol").agg(
    avg_volatility=("VolatilityScore", "mean"),
    max_volatility=("VolatilityScore", "max"),
    avg_momentum=("MomentumScore", "mean"),
    avg_volume=("VolumeScore", "mean"),
    avg_volume_spike=("VolumeSpike", "mean"),
    avg_change_pct=("regularMarketChangePercent", "mean")
)

In [40]:
final = price_stats.join(direction_changes).join(metrics)

# Normalize direction_changes
final["dir_change_norm"] = final["direction_changes"].rank(pct=True)

# Swing Score formula
final["swing_score"] = (
    final["range_pct"] * 0.35 +
    final["dir_change_norm"] * 0.30 +
    final["avg_volatility"] * 0.20 +
    final["avg_volume"] * 0.15
)

final = final.sort_values("swing_score", ascending=False)

In [41]:
swing_candidates = final[
    (final["avg_volatility"] > 0.6) &
    (final["avg_volume"] > 0.5) &
    (final["range_pct"] > 0.25)
].sort_values("swing_score", ascending=False)

top_swings = swing_candidates.head(20)

top_swings.head(10)

,min_price,max_price,mean_price,range_pct,direction_changes,avg_volatility,max_volatility,avg_momentum,avg_volume,avg_volume_spike,avg_change_pct,dir_change_norm,swing_score
symbol,,,,,,,,,,,,,
SNDK,202.4600,1562.3400,288.096882,4.720218,137,0.964159,1.000000,0.835794,0.749435,0.829592,7.274895,0.993596,2.255402
AAOI,29.3500,223.1000,41.888082,4.625421,3,0.772332,1.000000,0.938490,0.836719,1.212836,11.303639,0.854680,2.155275
NOW,88.4300,807.4500,165.623885,4.341282,37,0.817853,0.992126,0.556966,0.781467,0.726397,2.105953,0.959606,2.088121
MU,234.8050,928.4100,294.193758,2.357647,259,0.925227,1.000000,0.751820,0.801984,0.983750,4.957463,1.000000,1.430519
ARM,107.5800,410.7101,125.751514,2.410548,48,0.821064,0.992941,0.675212,0.849016,1.141438,4.545593,0.966749,1.425282
MRVL,86.7600,316.4300,97.852869,2.347095,25,0.851711,1.000000,0.791104,0.719886,1.258382,4.654758,0.937192,1.380966
DELL,118.1600,466.4900,138.042010,2.523362,1,0.903201,1.000000,0.839582,0.907422,1.900456,6.143690,0.417241,1.325103
BE,80.7100,315.1100,108.840836,2.153603,25,0.904179,0.997647,0.905011,0.671529,0.742974,8.310092,0.937192,1.316484
INTC,35.7099,124.9200,42.504120,2.098858,149,0.702141,0.955224,0.813089,0.715343,0.790486,6.253726,0.994828,1.280778


In [42]:
import plotly.express as px
import os

# -----------------------------
# TOP 20 ONLY (most important change)
# -----------------------------
df_plot = swing_candidates.head(20).copy()

# ensure clean ordering (best first visually)
df_plot = df_plot.sort_values("swing_score", ascending=False)

fig = px.scatter(
    df_plot,
    x="avg_volume_spike",
    y="avg_change_pct",
    size="range_pct",
    color="avg_volatility",
    text=df_plot.index,   # show tickers directly
    hover_name=df_plot.index,
    color_continuous_scale="Blues",
    size_max=30
)

# -----------------------------
# Make it modern + readable
# -----------------------------
fig.update_traces(
    textposition="top center",
    marker=dict(line=dict(width=1, color="white"))
)

fig.update_layout(
    title="Top 20 Swing Stocks (High-Confidence Set)",
    xaxis_title="Volume Spike",
    yaxis_title="Price Change %",
    
    plot_bgcolor="#0b0f1a",
    paper_bgcolor="#0b0f1a",
    font_color="white",

    height=650,
    margin=dict(l=40, r=40, t=60, b=40)
)
 
chart_path = os.path.join(OUTPUT_DIR, "top_20_swing_radar.html")
fig.write_html(chart_path, include_plotlyjs="cdn")
 

In [43]:
import plotly.express as px
import os

df_bar = swing_candidates.head(100).copy()

# ensure symbol exists (sometimes it's index)
df_bar = df_bar.reset_index()

if "index" in df_bar.columns:
    df_bar = df_bar.rename(columns={"index": "symbol"})

# sort for proper ranking display
df_bar = df_bar.sort_values("swing_score", ascending=True)

fig = px.bar(
    df_bar,
    y="symbol",
    x="swing_score",
    orientation="h",
    color="avg_volatility",
    color_continuous_scale="Greens",
    hover_data=[
        "range_pct",
        "direction_changes",
        "avg_volume",
        "avg_volume_spike"
    ],
    text="swing_score"
)

fig.update_traces(texttemplate="%{text:.2f}", textposition="outside")

fig.update_layout(
    title="Top 100 Swing Candidates (Ranked by Swing Score)",
    xaxis_title="Swing Score",
    yaxis_title="Symbol",
    plot_bgcolor="black",
    paper_bgcolor="black",
    font_color="white",
    height=900
) 
 
chart_paths = os.path.join(OUTPUT_DIR, "swing_candidates_bar.html")
fig.write_html(chart_paths, include_plotlyjs="cdn")

In [44]:
import plotly.express as px
import plotly.graph_objects as go
import os

df_bar = swing_candidates.head(100).copy().reset_index()

if "index" in df_bar.columns:
    df_bar = df_bar.rename(columns={"index": "symbol"})

df_bar = df_bar.sort_values("swing_score", ascending=True)

# 90th percentile threshold
threshold = df_bar["swing_score"].quantile(0.9)

fig = px.bar(
    df_bar,
    y="symbol",
    x="swing_score",
    orientation="h",
    color="avg_volatility",
    color_continuous_scale="Blues",  # 🔵 DARK BLUE SCALE
    hover_data=["range_pct", "direction_changes", "avg_volume"]
)

# Threshold line
fig.add_vline(
    x=threshold,
    line_width=2,
    line_dash="dash",
    line_color="white"
)

fig.add_annotation(
    x=threshold,
    y=1.02,
    xref="x",
    yref="paper",
    text="90th percentile",
    showarrow=False,
    font=dict(color="white")
)

# Dark theme styling
fig.update_layout(
    title="Top Swing Candidates (Dark Blue Theme)",
    xaxis_title="Swing Score",
    yaxis_title="Symbol",
    plot_bgcolor="#0b0f1a",   # deep navy background
    paper_bgcolor="#0b0f1a",
    font_color="white",
    height=900
)
 

chart_pathx = os.path.join(OUTPUT_DIR, "swing_bar_dark_blue.html")
fig.write_html(chart_pathx, include_plotlyjs="cdn")